In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

FULL_LABELED_PATH = "./data/full_labeled.csv"
TEST_PATH = "./data/test.csv"

In [2]:
full = pd.read_csv(FULL_LABELED_PATH)
test = pd.read_csv(TEST_PATH)

# Normalize column names
full = full.rename(columns={c: c.lower() for c in full.columns})
test = test.rename(columns={c: c.lower() for c in test.columns})

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace('"', "")
    x = re.sub(r"\s+", " ", x)
    return x

# Create robust matching keys
for df in [full, test]:
    df["name_key"] = df["name"].map(clean_text)
    df["sex_key"] = df["sex"].map(clean_text)
    df["ticket_key"] = df["ticket"].map(clean_text)

    df["age_round"] = pd.to_numeric(df["age"], errors="coerce").round(4)
    df["fare_round"] = pd.to_numeric(df["fare"], errors="coerce").round(4)

# Strong matching key
match_keys = [
    "name_key",
    "pclass",
    "sex_key",
    "sibsp",
    "parch",
    "ticket_key",
]

# Check for duplicate keys in full labeled data
dupes = full[full.duplicated(match_keys, keep=False)]
if len(dupes) > 0:
    print("Warning: duplicate match keys found in full_labeled:")
    display(dupes[["name", "pclass", "sex", "age", "sibsp", "parch", "ticket", "survived"]])

matched = test.merge(
    full[match_keys + ["survived"]],
    on=match_keys,
    how="left",
    validate="one_to_one"
)

missing = matched[matched["survived"].isna()]
print(f"Matched rows: {matched['survived'].notna().sum()} / {len(test)}")

if len(missing) > 0:
    print("Unmatched rows:")
    display(missing[["passengerid", "name", "pclass", "sex", "age", "sibsp", "parch", "ticket", "fare"]])
else:
    print("All test rows matched successfully.")

# Local truth file for evaluation only — do not submit this to Kaggle.
local_truth = matched[["passengerid", "survived"]].copy()
local_truth.columns = ["PassengerId", "Survived"]

local_truth.to_csv("best.csv", index=False)
local_truth.head()

Matched rows: 418 / 418
All test rows matched successfully.


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
